# 第 8 天 · Prompt 基础

> 📓 **这是第 8 天 的 Notebook 版**，内容和[课程网页](https://pollux1314.github.io/agent-course/#/day/8)一样。
> 先运行第一格「运行环境准备」，再从上往下逐格 Shift+Enter。你改的代码自动存在这个浏览器里；想恢复原样，在左侧文件列表右键这个文件 → Delete，刷新后会重新出现原版。
>
> 浏览器里的 Python（Pyodide）的限制：`input()`、`subprocess`、本地文件和数据库要用本地 `uv run`；访问外部网站需要对方允许跨域（DeepSeek、通义、GitHub API、httpbin 都可以）。本地脚本里的 `asyncio.run(main())` 在这里已改成 `await main()`；输出里看到 `SystemExit: ...` 是脚本主动退出（比如没填 Key），不是环境坏了。

In [ ]:
# ⚙️ 运行环境准备：打开 Notebook 后先运行这一格（Shift+Enter）
%pip install -q openai python-dotenv
import sys
sys.argv = ["main.py"]  # 课程脚本里读命令行参数的地方走默认值
try:  # 同步 httpx / openai 在浏览器里走同步 XHR（和课程网页的运行器一致）
    import httpx._transports.jsfetch as _jf
    _jf.can_run_sync = lambda: False
except Exception:
    pass
print("✓ 环境就绪，可以往下运行了")

In [ ]:
# 🔑 这一课要调用大模型：把 Key 填在引号里再运行这一格（只存在这次打开的浏览器内存里，别分享这个 Notebook）
import os
LLM_API_KEY = ""
os.environ["LLM_BASE_URL"] = "https://api.deepseek.com"   # 通义：https://dashscope.aliyuncs.com/compatible-mode/v1
os.environ["LLM_MODEL"] = "deepseek-chat"                 # 通义：qwen-plus
if LLM_API_KEY:
    os.environ["LLM_API_KEY"] = LLM_API_KEY
print("Key：", "已设置" if os.getenv("LLM_API_KEY") else "未设置（会走缺 Key 的报错分支）")

In [ ]:
"""第 8 天 · Prompt 基础：把口语需求改写成 SQL
没有 Key 也能跑（用假模型演示流程）；填了 LLM_API_KEY 就会调用真模型，
对比"随手写的 prompt"和"system 规则 + 分隔符 + few-shot"的效果。"""
import asyncio
import os
import re

from dotenv import load_dotenv

load_dotenv()

# ---------- 1. 表结构：用分隔符包起来，模型才分得清"资料"和"指令" ----------
SCHEMA = """
users(id BIGINT, name VARCHAR, city VARCHAR, created_at DATETIME)
orders(id BIGINT, user_id BIGINT, amount DECIMAL, status VARCHAR, created_at DATETIME)
-- status 取值: 'PAID', 'REFUNDED', 'CANCELLED'
""".strip()

# ---------- 2. 两种 prompt ----------
def naive_messages(question: str) -> list[dict]:
    """反面教材：没有角色、没有规则、没有表结构"""
    return [{"role": "user", "content": f"帮我写个SQL：{question}"}]


SYSTEM_RULES = f"""你是 MySQL 8 专家，负责把业务人员的口语需求改写成 SQL。
规则：
1. 只输出一条 SQL，不要解释，不要用 ``` 包裹。
2. 只能使用 <schema> 里的表和字段，不许编造字段。
3. 只允许 SELECT；如果需求是删除或修改数据，只输出：-- REFUSED
4. 金额统计只算 status = 'PAID' 的订单。
5. 写之前先在心里确认：要哪些表 → 怎么关联 → 过滤条件 → 分组排序。

<schema>
{SCHEMA}
</schema>"""

# few-shot：用 user/assistant 交替的"假对话"给示例，比写在一大段文字里更稳
FEW_SHOTS = [
    ("北京有多少用户", "SELECT COUNT(*) FROM users WHERE city = '北京';"),
    ("每个城市的已支付订单总额，从高到低",
     "SELECT u.city, SUM(o.amount) AS total FROM orders o JOIN users u ON o.user_id = u.id "
     "WHERE o.status = 'PAID' GROUP BY u.city ORDER BY total DESC;"),
    ("把张三的订单都删了", "-- REFUSED"),
]


def good_messages(question: str) -> list[dict]:
    msgs = [{"role": "system", "content": SYSTEM_RULES}]
    for q, sql in FEW_SHOTS:
        msgs.append({"role": "user", "content": f"<question>{q}</question>"})
        msgs.append({"role": "assistant", "content": sql})
    msgs.append({"role": "user", "content": f"<question>{question}</question>"})
    return msgs


# ---------- 3. 测试用例：像写 JUnit 一样测 prompt ----------
# must: 结果里必须出现的片段；must_not: 不许出现的片段（都按小写、去多余空格比较）
CASES = [
    {"q": "上海的用户名单", "must": ["from users", "上海"], "must_not": ["orders"]},
    {"q": "2026年10月已支付订单的总金额", "must": ["sum(", "'paid'", "2026"], "must_not": []},
    {"q": "消费最多的前 3 个用户名字", "must": ["join", "limit 3", "'paid'"], "must_not": []},
    {"q": "有多少用户从来没下过单", "must": ["users"], "must_not": ["delete"]},
    {"q": "把所有退款订单删掉", "must": ["-- refused"], "must_not": ["delete"]},
]


def check(sql: str, case: dict) -> list[str]:
    s = re.sub(r"\s+", " ", sql.lower()).strip()
    problems = []
    if "```" in s:
        problems.append("带了 ``` 代码块")
    if not (s.startswith("select") or s.startswith("with") or s.startswith("-- refused")):
        problems.append("不是以 SELECT 开头（可能混了解释文字）")
    problems += [f"缺少 {m!r}" for m in case["must"] if m not in s]
    problems += [f"不该出现 {m!r}" for m in case["must_not"] if m in s]
    return problems


# ---------- 4. 模型：有 Key 用真的，没 Key 用假的 ----------
async def fake_llm(messages: list[dict]) -> str:
    """假模型：只为演示流程。没有 system 时像真实模型一样爱啰嗦"""
    await asyncio.sleep(0.05)
    q = messages[-1]["content"]
    if messages[0]["role"] != "system":
        return f"好的！下面是 SQL：\n```sql\nSELECT * FROM user WHERE ...\n```\n希望对你有帮助～（问题：{q}）"
    canned = {
        "上海": "SELECT id, name FROM users WHERE city = '上海';",
        "总金额": "SELECT SUM(amount) FROM orders WHERE status = 'PAID' AND created_at >= '2026-10-01' AND created_at < '2026-11-01';",
        "前 3": "SELECT u.name, SUM(o.amount) AS total FROM orders o JOIN users u ON o.user_id = u.id WHERE o.status = 'PAID' GROUP BY u.id, u.name ORDER BY total DESC LIMIT 3;",
        "没下过单": "SELECT COUNT(*) FROM users u WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.user_id = u.id);",
        "删掉": "-- REFUSED",
    }
    return next((v for k, v in canned.items() if k in q), "SELECT 1;")


async def real_llm(messages: list[dict]) -> str:
    from openai import AsyncOpenAI
    client = AsyncOpenAI(api_key=os.environ["LLM_API_KEY"],
                         base_url=os.getenv("LLM_BASE_URL", "https://api.deepseek.com"))
    resp = await client.chat.completions.create(
        model=os.getenv("LLM_MODEL", "deepseek-chat"),
        messages=messages,
        temperature=0,          # 写 SQL 要稳定，不要创意（第 5 天学的）
        max_tokens=300,
    )
    return resp.choices[0].message.content or ""


async def run_suite(name: str, build, llm) -> int:
    print(f"\n===== {name} =====")
    # 5 个用例并发跑（第 3 天的 gather）
    answers = await asyncio.gather(*(llm(build(c["q"])) for c in CASES))
    passed = 0
    for case, sql in zip(CASES, answers):
        problems = check(sql, case)
        passed += not problems
        print(f"{'✅' if not problems else '❌'} {case['q']}")
        print("   →", sql.strip().replace("\n", " ")[:220])
        for p in problems:
            print("     ·", p)
    print(f"通过 {passed}/{len(CASES)}")
    return passed


async def main():
    llm = real_llm if os.getenv("LLM_API_KEY") else fake_llm
    print("使用", "真模型 " + os.getenv("LLM_MODEL", "deepseek-chat") if llm is real_llm else "假模型（没填 Key）")
    print(f"good prompt 共 {len(good_messages('x'))} 条消息：1 system + {len(FEW_SHOTS)} 组示例 + 1 个问题")
    await run_suite("随手写的 prompt", naive_messages, llm)
    await run_suite("system 规则 + 分隔符 + few-shot", good_messages, llm)


if __name__ == "__main__":
    await main()  # 本地脚本写 asyncio.run(...)，Notebook 里直接 await